In [0]:
# src/credit_cards/balance.py

from pyspark.sql import functions as F

def calculate_statement_balance(df):

    result = (
        df.groupBy("account_id")
          .agg(
              F.sum("amount").alias("statement_balance"),
              F.count("*").alias("txn_count")
          )
    )

    return result

In [0]:
from pyspark.testing.utils import assertDataFrameEqual
# from credit_cards.balance import calculate_statement_balance

def test_statement_balance():

    df = spark.createDataFrame(
        [
            (1001, 200),
            (1001, -100),
            (1002, 500)
        ],
        ["account_id", "amount"]
    )

    result = calculate_statement_balance(df)

    expected = spark.createDataFrame(
        [
            (1001, 100, 2),
            (1002, 500, 1)
        ],
        ["account_id", "statement_balance", "txn_count"]
    )

    assertDataFrameEqual(
        result.orderBy("account_id"),
        expected.orderBy("account_id")
    )

    print("✅ test_statement_balance passed")

In [0]:
test_statement_balance()

✅ test_statement_balance passed


In [0]:
def run_all_tests():
    test_statement_balance()
    print("🎯 All tests passed")

run_all_tests()

✅ test_statement_balance passed
🎯 All tests passed


In [0]:
from pyspark.sql.types import *
from pyspark.sql import Row
from decimal import Decimal
from datetime import date

schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("amount", DecimalType(10,2), True),
    StructField("txn_date", DateType(), True),
    StructField("merchant_code", StringType(), True)
])

data_day1 = [
    Row("T1001", "A2001", Decimal("120.50"), date(2026, 2, 15), "M001"),
    Row("T1002", "A2002", Decimal("85.20"),  date(2026, 2, 15), "M002"),
    Row("T1003", "A2003", Decimal("300.00"), date(2026, 2, 15), "M003")
]

df_day1 = spark.createDataFrame(data_day1, schema)

df_day1.printSchema()
df_day1.show()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- merchant_code: string (nullable = true)

+------+----------+------+----------+-------------+
|txn_id|account_id|amount|  txn_date|merchant_code|
+------+----------+------+----------+-------------+
| T1001|     A2001|120.50|2026-02-15|         M001|
| T1002|     A2002| 85.20|2026-02-15|         M002|
| T1003|     A2003|300.00|2026-02-15|         M003|
+------+----------+------+----------+-------------+



In [0]:
schema_day2 = StructType([
    StructField("txn_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("amount", StringType(), True),   # ❌ 错误
    StructField("txn_date", DateType(), True),
    StructField("merchant_code", StringType(), True),
    StructField("currency", StringType(), True)  # ❌ 新列
])

data_day2 = [
    Row("T2001", "A2001", "150.75", date(2026, 2, 16), "M001", "CAD")
]

df_day2 = spark.createDataFrame(data_day2, schema_day2)

df_day2.printSchema()

root
 |-- txn_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- txn_date: date (nullable = true)
 |-- merchant_code: string (nullable = true)
 |-- currency: string (nullable = true)



In [0]:
from pyspark.testing.utils import assertSchemaEqual

assertSchemaEqual(df_day1.schema, df_day2.schema)

---------------------------------------------------------------------------
PySparkAssertionError                     Traceback (most recent call last)
File <command-5915642027683794>, line 3
      1 from pyspark.testing.utils import assertSchemaEqual
----> 3 assertSchemaEqual(df_day1.schema, df_day2.schema)

File /databricks/spark/python/pyspark/instrumentation_utils.py:47, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     45 start = time.perf_counter()
     46 try:
---> 47     res = func(*args, **kwargs)
     48     logger.log_success(
     49         module_name, class_name, function_name, time.perf_counter() - start, signature
     50     )
     51     return res

File /databricks/spark/python/pyspark/testing/utils.py:556, in assertSchemaEqual(actual, expected, ignoreNullable, ignoreColumnOrder, ignoreColumnName)
    554 generated_diff = difflib.ndiff(str(actual).splitlines(), str(expected).splitlines())
    555 error_msg = "\n".join(generated_diff)
--> 556 raise PySparkAsse

In [0]:
source_cols = set(df_day2.dtypes)
target_cols = set(df_day1.dtypes)

missing_cols = target_cols - source_cols
new_cols = source_cols - target_cols

print("Missing:", missing_cols)
print("New:", new_cols)

Missing: {('amount', 'decimal(10,2)')}
New: {('currency', 'string'), ('amount', 'string')}


In [0]:
df = spark.createDataFrame(
        [
            (1001, 200),
            (1001, -100),
            (1002, 500)
        ],
        ["account_id", "amount"]
    )
df.printSchema()
df.show(5)
df.explain("formatted")

root
 |-- account_id: long (nullable = true)
 |-- amount: long (nullable = true)

+----------+------+
|account_id|amount|
+----------+------+
|      1001|   200|
|      1001|  -100|
|      1002|   500|
+----------+------+

== Physical Plan ==
*(1) Scan ExistingRDD[account_id#236L,amount#237L]




In [0]:
df.rdd.getNumPartitions()

4

In [0]:
breakpoint()